In [ ]:
!pip install -q transformers accelerate torch

In [ ]:
import random
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


# =========================
# Configuration
# =========================

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

LOCAL_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

TEMPERATURE = 0.6
TOP_P = 0.9
MAX_NEW_TOKENS = 160
REPETITION_PENALTY = 1.05
MAX_RETRIES_PER_TASK = 3

OUTPUT_FILENAME = "final_synthetic_dataset_local_llm.jsonl"
ERROR_FILENAME = "generation_errors_local_llm.jsonl"


# =========================
# Load Local LLM
# =========================

print(f"Loading local model: {LOCAL_MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model.eval()

print("Model loaded successfully.")


# =========================
# Risk Factor Instructions
# =========================

def get_risk_factor_instruction(risk_factors):
    """
    Adds concrete symptom-level guidance for each target risk factor.
    This helps the local LLM stay aligned with the intended labels.
    """

    mapping = {
        "Chest_Pain_or_Pressure": (
            "- For Chest_Pain_or_Pressure: describe chest pressure, heaviness, tightness, or squeezing. "
            "Do not use diagnostic names."
        ),
        "Respiratory_Distress": (
            "- For Respiratory_Distress: describe trouble breathing, shortness of breath, wheezing, "
            "or inability to catch breath."
        ),
        "Acute_Neurological": (
            "- For Acute_Neurological: describe sudden weakness, numbness, face drooping, confusion, "
            "trouble speaking, or sudden severe dizziness."
        ),
        "Severe_Infection": (
            "- For Severe_Infection: describe fever, chills, worsening wound, pus, spreading redness, "
            "or feeling very ill."
        ),
        "Anaphylaxis_or_Allergy": (
            "- For Anaphylaxis_or_Allergy: describe rapid swelling of lips, face, or throat, widespread rash, "
            "or breathing trouble after food, medication, or insect exposure. Do not use the word anaphylaxis."
        ),
        "Uncontrolled_Bleeding": (
            "- For Uncontrolled_Bleeding: describe bleeding that keeps soaking bandages, will not slow down, "
            "or restarts repeatedly."
        ),
        "Severe_Pain": (
            "- For Severe_Pain: describe intense pain in a non-chest location such as abdomen, back, tooth, ear, "
            "joint, or leg. Avoid chest pressure, squeezing chest pain, trouble breathing, fainting, or neurological symptoms."
        ),
        "Trauma_or_Injury": (
            "- For Trauma_or_Injury: describe a fall, cut, sprain, burn, accident, or injury with pain, swelling, "
            "bruising, or reduced movement."
        ),
        "Medication_Adverse_Reaction": (
            "- For Medication_Adverse_Reaction: mention that symptoms started after a new medication, dose change, "
            "missed dose, or combining medications. Use symptoms like nausea, shakiness, rash, dizziness, sleepiness, "
            "or stomach upset."
        ),
        "None": (
            "- For None: do not include risk-factor symptoms. Prefer routine requests, mild stable symptoms, "
            "prescription refill, lab results, forms, or appointment scheduling."
        )
    }

    instructions = []
    for rf in risk_factors:
        if rf in mapping:
            instructions.append(mapping[rf])

    return "\n".join(instructions)


# =========================
# Step A: Prompt Builder
# =========================

def build_generation_prompt(task):
    valid_urgencies = {"Green", "Yellow", "Red"}
    valid_noise_levels = {"low", "medium", "high"}

    required_keys = [
        "urgency_level",
        "risk_factors",
        "insufficient_information",
        "age_group",
        "writing_style",
        "anxiety_level",
        "noise_level"
    ]

    for key in required_keys:
        if key not in task:
            raise KeyError(f"Missing required key in task: {key}")

    if task["urgency_level"] not in valid_urgencies:
        raise ValueError(f"Invalid urgency_level: {task['urgency_level']}")

    if task["noise_level"] not in valid_noise_levels:
        raise ValueError(f"Invalid noise_level: {task['noise_level']}")

    if not isinstance(task["risk_factors"], list):
        raise TypeError("risk_factors must be a list of strings")

    if "None" in task["risk_factors"] and len(task["risk_factors"]) > 1:
        raise ValueError("'None' cannot appear together with other risk factors")

    if not isinstance(task["insufficient_information"], bool):
        raise TypeError("insufficient_information must be True or False")

    risk_factors_str = ", ".join(task["risk_factors"])

    system_prompt = (
        "You generate synthetic patient-written portal messages for a machine learning classification dataset.\n\n"
        "CRITICAL RULES:\n"
        "1. WRITE ONLY THE PATIENT MESSAGE. Do not include any introduction, formatting, labels, or quotes.\n"
        "2. DO NOT include diagnostic names, for example do not say 'heart attack', 'stroke', or 'anaphylaxis'.\n"
        "3. DO NOT output formal medical jargon unless the persona realistically would use it.\n"
        "4. DO NOT mention the target labels anywhere in the text.\n"
        "5. DO NOT include professional advice, diagnosis, or triage decisions.\n"
        "6. Do not mention ER, emergency room, ambulance, hospital now, or urgent care.\n"
        "7. The patient may ask questions like 'Should I worry?' or 'Can someone call me?'.\n"
        "8. The output must look like a raw, unedited, human-written message typed into a healthcare app.\n"
        "9. Keep the message concise: between 1 and 5 sentences.\n"
        "10. The message must be entirely in English."
    )

    if task["insufficient_information"]:
        info_instruction = (
            "- The message should be vague, ambiguous, or incomplete in at least one important way. "
            "For example, it may omit when the problem started, how severe it is, whether it is getting worse, "
            "or which exact body part is affected. It should be hard to safely route without human follow-up."
        )
    else:
        info_instruction = (
            "- The message should contain enough concrete detail for a human reviewer to understand the basic context, "
            "such as symptom type, timing, severity, or relevant background."
        )

    pain_green_instruction = ""
    if "Severe_Pain" in task["risk_factors"] and task["urgency_level"] == "Green":
        pain_green_instruction = (
            "- Special context: Since the urgency is Green but Severe_Pain is flagged, frame this as a long-standing "
            "or chronic issue, such as recurring back pain, arthritis, or an old injury. The patient may mention high "
            "pain levels but should mainly be asking for a routine administrative action, such as a refill, follow-up, "
            "or appointment scheduling."
        )

    if task["noise_level"] == "low":
        noise_instruction = (
            "- Noise level is low: the message should be mostly clear, with at most minor informality."
        )
    elif task["noise_level"] == "medium":
        noise_instruction = (
            "- Noise level is medium: include some informal phrasing, minor typos, missing punctuation, "
            "or small irrelevant details."
        )
    else:
        noise_instruction = (
            "- Noise level is high: use messy wording, fragments, typos, repeated thoughts, missing punctuation, "
            "or irrelevant details."
        )

    if task["urgency_level"] == "Green":
        urgency_instruction = (
            "- For Green urgency: the message should be routine, administrative, chronic/stable, or non-urgent. "
            "Avoid alarming symptoms such as chest pain, trouble breathing, fainting, sudden weakness, numbness, "
            "seizure-like events, uncontrolled bleeding, rapid swelling, severe sudden pain, confusion, or high fever. "
            "Good Green examples include prescription refill, routine appointment, test result request, stable chronic issue, "
            "or mild symptoms without worrying features."
        )
    elif task["urgency_level"] == "Yellow":
        urgency_instruction = (
            "- For Yellow urgency: the message should describe a concerning but not immediately life-threatening issue. "
            "It may require same-day human review, but should not sound like an obvious emergency. "
            "Avoid collapse, severe breathing difficulty, uncontrolled bleeding, loss of consciousness, or clear sudden neurological deficits."
        )
    else:
        urgency_instruction = (
            "- For Red urgency: the message should contain clear warning signs that justify immediate human review. "
            "Use patient-like descriptions rather than diagnostic names. Do not explicitly say it is an emergency."
        )

    persona_instruction = ""
    if task["age_group"] == "child (parent writing)":
        persona_instruction = (
            "- Since the patient is a child, the message must be written by a parent or caregiver. "
            "Use phrases like 'my child', 'my son', or 'my daughter'. "
            "Do not write from the child's first-person perspective. "
            "Do not use phrases like 'Mommy', 'Daddy', or 'I want my mom'."
        )

    risk_factor_instruction = get_risk_factor_instruction(task["risk_factors"])

    user_prompt = f"""
Generate ONE patient portal message based on these exact constraints.

[TARGET ML LABELS]
- Urgency Level: {task["urgency_level"]}
- Detected Risk Factors: {risk_factors_str}
- Is Information Insufficient?: {task["insufficient_information"]}

[PATIENT PERSONA]
- Age Group / Context: {task["age_group"]}
- Writing Style: {task["writing_style"]}
- Anxiety Level: {task["anxiety_level"]}
- Noise Level: {task["noise_level"]}

[GENERATION GUIDELINES]
{info_instruction}
{pain_green_instruction}
{noise_instruction}
{urgency_instruction}
{persona_instruction}
{risk_factor_instruction}
- Match the requested writing style. For example:
  - 'panicked with typos' should sound frantic and imperfect.
  - 'calm but descriptive' should be readable and organized.
  - 'short and fragmented' should use brief, incomplete phrases.
  - 'verbose and disorganized' should include extra details or wandering structure.
- The message should sound like it was written by the patient or caregiver, not by a doctor.
- Do not reveal or directly name the labels.

Patient Message:
""".strip()

    return {
        "system_prompt": system_prompt,
        "user_prompt": user_prompt
    }


# =========================
# Step B: Generation Manifest
# =========================

generation_plan = [
    # Green total = 1200
    {"urgency": "Green", "risk_factors": ["None"], "insufficient": False, "count": 800},
    {"urgency": "Green", "risk_factors": ["None"], "insufficient": True, "count": 250},
    {"urgency": "Green", "risk_factors": ["Medication_Adverse_Reaction"], "insufficient": False, "count": 80},
    {"urgency": "Green", "risk_factors": ["Severe_Pain"], "insufficient": True, "count": 70},

    # Yellow total = 1050
    {"urgency": "Yellow", "risk_factors": ["Severe_Infection"], "insufficient": False, "count": 220},
    {"urgency": "Yellow", "risk_factors": ["Severe_Pain"], "insufficient": False, "count": 180},
    {"urgency": "Yellow", "risk_factors": ["Medication_Adverse_Reaction"], "insufficient": False, "count": 160},
    {"urgency": "Yellow", "risk_factors": ["Trauma_or_Injury"], "insufficient": False, "count": 160},
    {"urgency": "Yellow", "risk_factors": ["Respiratory_Distress"], "insufficient": True, "count": 100},
    {"urgency": "Yellow", "risk_factors": ["Chest_Pain_or_Pressure"], "insufficient": True, "count": 100},
    {"urgency": "Yellow", "risk_factors": ["None"], "insufficient": True, "count": 130},

    # Red total = 750
    {"urgency": "Red", "risk_factors": ["Acute_Neurological"], "insufficient": False, "count": 220},
    {"urgency": "Red", "risk_factors": ["Respiratory_Distress"], "insufficient": False, "count": 160},
    {"urgency": "Red", "risk_factors": ["Chest_Pain_or_Pressure"], "insufficient": False, "count": 150},
    {"urgency": "Red", "risk_factors": ["Anaphylaxis_or_Allergy"], "insufficient": False, "count": 100},
    {"urgency": "Red", "risk_factors": ["Uncontrolled_Bleeding"], "insufficient": False, "count": 80},

    # Multi-risk Red cases, total = 40
    {"urgency": "Red", "risk_factors": ["Chest_Pain_or_Pressure", "Respiratory_Distress"], "insufficient": False, "count": 15},
    {"urgency": "Red", "risk_factors": ["Trauma_or_Injury", "Uncontrolled_Bleeding"], "insufficient": False, "count": 15},
    {"urgency": "Red", "risk_factors": ["Severe_Infection", "Severe_Pain"], "insufficient": False, "count": 10},
]

age_groups = [
    "child (parent writing)",
    "young adult (18-35)",
    "adult (36-65)",
    "elderly (65+)"
]

writing_styles = [
    "panicked with typos",
    "calm but descriptive",
    "short and fragmented",
    "verbose and disorganized"
]

anxiety_levels = ["Low", "Medium", "High"]
noise_levels = ["low", "medium", "high"]


def create_generation_manifest():
    all_tasks = []

    for plan in generation_plan:
        for _ in range(plan["count"]):
            task = {
                "urgency_level": plan["urgency"],
                "risk_factors": plan["risk_factors"],
                "insufficient_information": plan["insufficient"],
                "age_group": random.choice(age_groups),
                "writing_style": random.choice(writing_styles),
                "anxiety_level": random.choice(anxiety_levels),
                "noise_level": random.choice(noise_levels)
            }
            all_tasks.append(task)

    random.shuffle(all_tasks)

    for i, task in enumerate(all_tasks):
        task["task_id"] = f"task_{i:05d}"

    return all_tasks


# =========================
# Step C: Local LLM Generation
# =========================

def generate_with_local_llm(system_prompt, user_prompt, max_new_tokens=MAX_NEW_TOKENS):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            repetition_penalty=REPETITION_PENALTY,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return clean_generated_text(generated_text)


def clean_generated_text(text):
    text = text.strip()

    prefixes_to_remove = [
        "Patient Message:",
        "patient message:",
        "Message:",
        "message:",
        "\"",
        "'"
    ]

    for prefix in prefixes_to_remove:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()

    text = text.strip("\"'")
    return text.strip()


# =========================
# Step D: Validation
# =========================

def validate_generated_message(message):
    if not message or not message.strip():
        return False, "empty_message"

    word_count = len(message.split())

    if word_count < 3:
        return False, "too_short"

    if word_count > 120:
        return False, "too_long"

    lower_msg = message.lower()

    forbidden_phrases = [
        "heart attack",
        "stroke",
        "anaphylaxis",
        "urgency level",
        "risk factor",
        "detected risk",
        "target label",
        "green urgency",
        "yellow urgency",
        "red urgency",
        "as an ai",
        "i am an ai",
        "patient message:",
        "[target ml labels]",
        "[patient persona]",
        "[generation guidelines]",

        # Avoid explicit triage / emergency-routing wording
        "go to the er",
        "take them to the er",
        "take him to the er",
        "take her to the er",
        "emergency room",
        "ambulance",
        "hospital now",
        "urgent care"
    ]

    for phrase in forbidden_phrases:
        if phrase in lower_msg:
            return False, f"forbidden_phrase:{phrase}"

    return True, "ok"


def validate_label_consistency(message, task):
    lower_msg = message.lower()

    red_flag_terms = [
        "chest hurts", "chest pain", "chest pressure", "pressure in my chest",
        "tightness in my chest", "can't breathe", "cannot breathe", "cant breathe",
        "short of breath", "trouble breathing", "hard to breathe", "difficulty breathing",
        "pass out", "passed out", "faint", "fainted", "left arm", "right arm",
        "arm feels numb", "arm is numb", "numb arm", "weak arm", "face drooping",
        "slurred speech", "can't speak", "cant speak", "confused", "suddenly confused",
        "bleeding won't stop", "bleeding wont stop", "uncontrollable bleeding",
        "swelling in my face", "throat is closing", "lips are swelling", "high fever",
        "very high fever", "seizure", "seizing", "convulsion", "convulsions",
        "lost consciousness", "unconscious", "not waking up", "won't wake up",
        "wont wake up", "blue lips", "turning blue", "limp", "not responding",
        "unresponsive"
    ]

    yellow_or_red_terms = [
        "fever", "infection", "pus", "swollen", "wound", "rash", "vomiting",
        "dizzy", "severe pain", "really bad pain", "can barely stand",
        "puffy face", "swollen face", "face is swollen", "toothache really bad",
        "really bad toothache"
    ]

    if task["urgency_level"] == "Green":
        for term in red_flag_terms:
            if term in lower_msg:
                return False, f"green_contains_red_flag:{term}"

    if task["risk_factors"] == ["None"]:
        for term in red_flag_terms + yellow_or_red_terms:
            if term in lower_msg:
                return False, f"none_risk_contains_risk_language:{term}"

    return True, "ok"


def validate_persona_consistency(message, task):
    lower_msg = message.lower()

    if task["age_group"] == "child (parent writing)":
        child_self_phrases = [
            "mommy", "daddy", "my mommy", "my daddy",
            "i want my mom", "i want my dad",
            "can you tell my mom", "can you tell my dad"
        ]

        for phrase in child_self_phrases:
            if phrase in lower_msg:
                return False, f"child_parent_writer_mismatch:{phrase}"

    return True, "ok"


# =========================
# Step E: Dataset Generation Loop
# =========================

def generate_dataset(limit=None):
    """
    Generate the synthetic dataset using a local LLM.

    Each task gets up to MAX_RETRIES_PER_TASK attempts.
    If all attempts fail validation, the task is saved to the error file.
    """

    all_tasks = create_generation_manifest()

    if limit is not None:
        all_tasks = all_tasks[:limit]

    print(f"Created manifest with {len(all_tasks)} randomized tasks.")
    print("Starting local LLM generation loop.")
    print(f"Max retries per task: {MAX_RETRIES_PER_TASK}")

    valid_count = 0
    failed_count = 0
    total_generation_attempts = 0

    with open(OUTPUT_FILENAME, "w", encoding="utf-8") as outfile, \
         open(ERROR_FILENAME, "w", encoding="utf-8") as errfile:

        for index, task in enumerate(all_tasks):
            success = False
            last_error_entry = None

            for attempt in range(1, MAX_RETRIES_PER_TASK + 1):
                total_generation_attempts += 1

                try:
                    prompts = build_generation_prompt(task)

                    generated_message = generate_with_local_llm(
                        prompts["system_prompt"],
                        prompts["user_prompt"]
                    )

                    is_valid, validation_reason = validate_generated_message(generated_message)

                    if is_valid:
                        is_valid, validation_reason = validate_label_consistency(
                            generated_message,
                            task
                        )

                    if is_valid:
                        is_valid, validation_reason = validate_persona_consistency(
                            generated_message,
                            task
                        )

                    if not is_valid:
                        last_error_entry = {
                            "task_id": task["task_id"],
                            "error_type": "validation_failed",
                            "validation_reason": validation_reason,
                            "generated_message": generated_message,
                            "attempt": attempt,
                            "task": task
                        }
                        continue

                    dataset_entry = {
                        "task_id": task["task_id"],
                        "text": generated_message,
                        "labels": {
                            "urgency_level": task["urgency_level"],
                            "risk_factors": task["risk_factors"],
                            "insufficient_information": task["insufficient_information"]
                        },
                        "metadata": {
                            "age_group": task["age_group"],
                            "writing_style": task["writing_style"],
                            "anxiety_level": task["anxiety_level"],
                            "noise_level": task["noise_level"],
                            "generation_model": LOCAL_MODEL_NAME,
                            "generation_method": "local_huggingface_llm",
                            "temperature": TEMPERATURE,
                            "top_p": TOP_P,
                            "successful_attempt": attempt
                        }
                    }

                    outfile.write(json.dumps(dataset_entry, ensure_ascii=False) + "\n")
                    outfile.flush()

                    valid_count += 1
                    success = True
                    break

                except Exception as e:
                    last_error_entry = {
                        "task_id": task.get("task_id"),
                        "error_type": "runtime_error",
                        "error_message": str(e),
                        "attempt": attempt,
                        "task": task
                    }
                    continue

            if not success:
                failed_count += 1

                if last_error_entry is None:
                    last_error_entry = {
                        "task_id": task.get("task_id"),
                        "error_type": "unknown_failure",
                        "task": task
                    }

                errfile.write(json.dumps(last_error_entry, ensure_ascii=False) + "\n")
                errfile.flush()

            if (index + 1) % 50 == 0:
                print(
                    f"Progress: {index + 1}/{len(all_tasks)} tasks processed | "
                    f"valid: {valid_count} | failed after retries: {failed_count} | "
                    f"total attempts: {total_generation_attempts}"
                )

    print("Finished.")
    print(f"Valid dataset entries saved to: {OUTPUT_FILENAME}")
    print(f"Failed tasks after retries saved to: {ERROR_FILENAME}")
    print(f"Valid examples: {valid_count}")
    print(f"Failed after retries: {failed_count}")
    print(f"Total generation attempts: {total_generation_attempts}")

# Run
generate_dataset()



Loading local model: Qwen/Qwen2.5-1.5B-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully.
Created manifest with 3000 randomized tasks.
Starting local LLM generation loop.
Max retries per task: 3
Progress: 50/3000 tasks processed | valid: 48 | failed after retries: 2 | total attempts: 74
Progress: 100/3000 tasks processed | valid: 97 | failed after retries: 3 | total attempts: 146
Progress: 150/3000 tasks processed | valid: 142 | failed after retries: 8 | total attempts: 226
Progress: 200/3000 tasks processed | valid: 189 | failed after retries: 11 | total attempts: 306
Progress: 250/3000 tasks processed | valid: 235 | failed after retries: 15 | total attempts: 379
Progress: 300/3000 tasks processed | valid: 283 | failed after retries: 17 | total attempts: 450
Progress: 350/3000 tasks processed | valid: 329 | failed after retries: 21 | total attempts: 524
Progress: 400/3000 tasks processed | valid: 377 | failed after retries: 23 | total attempts: 588
Progress: 450/3000 tasks processed | valid: 423 | failed after retries: 27 | total attempts: 656
Pr

In [4]:
import json

with open("final_synthetic_dataset_local_llm.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(json.loads(line))
        if i == 7:
            break

{'task_id': 'task_00000', 'text': "My child has been feeling really tired lately. They seem more sleepy than usual and don't want to play as much. Should I worry?", 'labels': {'urgency_level': 'Green', 'risk_factors': ['None'], 'insufficient_information': True}, 'metadata': {'age_group': 'child (parent writing)', 'writing_style': 'short and fragmented', 'anxiety_level': 'High', 'noise_level': 'high', 'generation_model': 'Qwen/Qwen2.5-1.5B-Instruct', 'generation_method': 'local_huggingface_llm', 'temperature': 0.6, 'top_p': 0.9, 'successful_attempt': 1}}
{'task_id': 'task_00001', 'text': "I'm feeling a bit tired and weak lately, especially after doing some light housework. I've been taking my blood pressure medication regularly, but I haven't had any major changes in my symptoms. Just wanted to check if this is normal or if I should be worried.", 'labels': {'urgency_level': 'Green', 'risk_factors': ['None'], 'insufficient_information': True}, 'metadata': {'age_group': 'elderly (65+)', '

In [5]:
import json
from collections import Counter
import statistics

DATASET_PATH = "final_synthetic_dataset_local_llm.jsonl"
ERROR_PATH = "generation_errors_local_llm.jsonl"

rows = []
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

errors = []
with open(ERROR_PATH, "r", encoding="utf-8") as f:
    for line in f:
        errors.append(json.loads(line))

print("=== Basic Counts ===")
print("Valid rows:", len(rows))
print("Failed rows:", len(errors))
print("Total tasks:", len(rows) + len(errors))
print("Success rate:", round(len(rows) / (len(rows) + len(errors)) * 100, 2), "%")

print("\n=== Urgency Distribution ===")
urgency_counts = Counter(row["labels"]["urgency_level"] for row in rows)
for label, count in urgency_counts.items():
    print(label, count, round(count / len(rows) * 100, 2), "%")

print("\n=== Risk Factor Distribution ===")
risk_counts = Counter()
for row in rows:
    for rf in row["labels"]["risk_factors"]:
        risk_counts[rf] += 1

for label, count in risk_counts.most_common():
    print(label, count, round(count / len(rows) * 100, 2), "%")

print("\n=== Insufficient Information Distribution ===")
insufficient_counts = Counter(row["labels"]["insufficient_information"] for row in rows)
for label, count in insufficient_counts.items():
    print(label, count, round(count / len(rows) * 100, 2), "%")

print("\n=== Text Length Statistics ===")
lengths_words = [len(row["text"].split()) for row in rows]
print("Min words:", min(lengths_words))
print("Max words:", max(lengths_words))
print("Mean words:", round(statistics.mean(lengths_words), 2))
print("Median words:", statistics.median(lengths_words))

print("\n=== Repeated Phrase Checks ===")
phrases = [
    "should i worry",
    "can someone call me",
    "should i call the doctor",
    "can you help me",
    "i'm worried",
    "not sure"
]

for phrase in phrases:
    count = sum(phrase in row["text"].lower() for row in rows)
    print(f"{phrase}: {count} ({round(count / len(rows) * 100, 2)}%)")

print("\n=== Successful Attempt Distribution ===")
attempt_counts = Counter(row["metadata"].get("successful_attempt", "missing") for row in rows)
for attempt, count in sorted(attempt_counts.items()):
    print(attempt, count, round(count / len(rows) * 100, 2), "%")

print("\n=== Error Reasons ===")
error_reasons = Counter(err.get("validation_reason", err.get("error_type", "unknown")) for err in errors)
for reason, count in error_reasons.most_common(20):
    print(reason, count)

=== Basic Counts ===
Valid rows: 2799
Failed rows: 201
Total tasks: 3000
Success rate: 93.3 %

=== Urgency Distribution ===
Green 1092 39.01 %
Yellow 988 35.3 %
Red 719 25.69 %

=== Risk Factor Distribution ===
None 1047 37.41 %
Respiratory_Distress 270 9.65 %
Chest_Pain_or_Pressure 255 9.11 %
Severe_Pain 253 9.04 %
Medication_Adverse_Reaction 238 8.5 %
Severe_Infection 224 8.0 %
Acute_Neurological 206 7.36 %
Trauma_or_Injury 156 5.57 %
Anaphylaxis_or_Allergy 96 3.43 %
Uncontrolled_Bleeding 92 3.29 %

=== Insufficient Information Distribution ===
True 602 21.51 %
False 2197 78.49 %

=== Text Length Statistics ===
Min words: 9
Max words: 110
Mean words: 40.12
Median words: 39

=== Repeated Phrase Checks ===
should i worry: 439 (15.68%)
can someone call me: 195 (6.97%)
should i call the doctor: 52 (1.86%)
can you help me: 185 (6.61%)
i'm worried: 461 (16.47%)
not sure: 140 (5.0%)

=== Successful Attempt Distribution ===
1 2068 73.88 %
2 548 19.58 %
3 183 6.54 %

=== Error Reasons ===
for

In [6]:
import json
import random
from collections import Counter
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)

DATASET_PATH = "final_synthetic_dataset_local_llm.jsonl"

TRAIN_PATH = "train.jsonl"
VAL_PATH = "validation.jsonl"
TEST_PATH = "test.jsonl"

# =========================
# Load dataset
# =========================

rows = []
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

print("Original rows:", len(rows))

# =========================
# Remove exact duplicate texts
# =========================

seen_texts = set()
deduped_rows = []

for row in rows:
    text_key = row["text"].strip().lower()
    if text_key not in seen_texts:
        seen_texts.add(text_key)
        deduped_rows.append(row)

rows = deduped_rows
print("After deduplication:", len(rows))

# =========================
# Define label spaces
# =========================

URGENCY_LABELS = ["Green", "Yellow", "Red"]

RISK_FACTOR_LABELS = [
    "Chest_Pain_or_Pressure",
    "Respiratory_Distress",
    "Acute_Neurological",
    "Severe_Infection",
    "Anaphylaxis_or_Allergy",
    "Uncontrolled_Bleeding",
    "Severe_Pain",
    "Trauma_or_Injury",
    "Medication_Adverse_Reaction",
    "None"
]

urgency_to_id = {label: i for i, label in enumerate(URGENCY_LABELS)}
risk_to_id = {label: i for i, label in enumerate(RISK_FACTOR_LABELS)}

# =========================
# Convert labels to model-ready format
# =========================

processed_rows = []

for row in rows:
    urgency = row["labels"]["urgency_level"]
    risk_factors = row["labels"]["risk_factors"]
    insufficient = row["labels"]["insufficient_information"]

    urgency_id = urgency_to_id[urgency]

    risk_vector = [0] * len(RISK_FACTOR_LABELS)
    for rf in risk_factors:
        risk_vector[risk_to_id[rf]] = 1

    processed_row = {
        "task_id": row["task_id"],
        "text": row["text"],
        "labels": {
            "urgency_level": urgency,
            "urgency_id": urgency_id,
            "risk_factors": risk_factors,
            "risk_vector": risk_vector,
            "insufficient_information": insufficient,
            "insufficient_id": int(insufficient)
        },
        "metadata": row.get("metadata", {})
    }

    processed_rows.append(processed_row)

print("Processed rows:", len(processed_rows))

# =========================
# Stratified split by urgency
# =========================

stratify_labels = [
    row["labels"]["urgency_level"]
    for row in processed_rows
]

train_rows, temp_rows = train_test_split(
    processed_rows,
    test_size=0.30,
    random_state=SEED,
    stratify=stratify_labels
)

temp_stratify = [
    row["labels"]["urgency_level"]
    for row in temp_rows
]

val_rows, test_rows = train_test_split(
    temp_rows,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_stratify
)

print("\nSplit sizes:")
print("Train:", len(train_rows))
print("Validation:", len(val_rows))
print("Test:", len(test_rows))

# =========================
# Save JSONL files
# =========================

def save_jsonl(path, data):
    with open(path, "w", encoding="utf-8") as f:
        for row in data:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

save_jsonl(TRAIN_PATH, train_rows)
save_jsonl(VAL_PATH, val_rows)
save_jsonl(TEST_PATH, test_rows)

print("\nSaved files:")
print(TRAIN_PATH)
print(VAL_PATH)
print(TEST_PATH)

# =========================
# Check distributions
# =========================

def print_distribution(name, split_rows):
    print(f"\n=== {name} ===")

    urgency_counts = Counter(row["labels"]["urgency_level"] for row in split_rows)
    print("Urgency:", urgency_counts)

    insufficient_counts = Counter(row["labels"]["insufficient_information"] for row in split_rows)
    print("Insufficient:", insufficient_counts)

    risk_counts = Counter()
    for row in split_rows:
        for rf in row["labels"]["risk_factors"]:
            risk_counts[rf] += 1

    print("Risk factors:")
    for label, count in risk_counts.most_common():
        print(f"  {label}: {count}")

print_distribution("Train", train_rows)
print_distribution("Validation", val_rows)
print_distribution("Test", test_rows)

Original rows: 2799
After deduplication: 2798
Processed rows: 2798

Split sizes:
Train: 1958
Validation: 420
Test: 420

Saved files:
train.jsonl
validation.jsonl
test.jsonl

=== Train ===
Urgency: Counter({'Green': 764, 'Yellow': 691, 'Red': 503})
Insufficient: Counter({False: 1534, True: 424})
Risk factors:
  None: 733
  Severe_Pain: 184
  Respiratory_Distress: 178
  Chest_Pain_or_Pressure: 174
  Medication_Adverse_Reaction: 166
  Severe_Infection: 165
  Acute_Neurological: 153
  Trauma_or_Injury: 97
  Uncontrolled_Bleeding: 67
  Anaphylaxis_or_Allergy: 66

=== Validation ===
Urgency: Counter({'Green': 163, 'Yellow': 149, 'Red': 108})
Insufficient: Counter({False: 324, True: 96})
Risk factors:
  None: 161
  Chest_Pain_or_Pressure: 50
  Respiratory_Distress: 48
  Severe_Pain: 35
  Medication_Adverse_Reaction: 33
  Trauma_or_Injury: 30
  Acute_Neurological: 25
  Severe_Infection: 21
  Anaphylaxis_or_Allergy: 15
  Uncontrolled_Bleeding: 10

=== Test ===
Urgency: Counter({'Green': 164, 'Y

In [8]:
!pip install -q transformers scikit-learn accelerate

In [9]:
import json
import numpy as np
import torch

from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)


# =========================
# Configuration
# =========================

MODEL_NAME = "distilbert-base-uncased"

TRAIN_PATH = "train.jsonl"
VAL_PATH = "validation.jsonl"
TEST_PATH = "test.jsonl"

OUTPUT_DIR = "./urgency_distilbert_model"

LABEL_NAMES = ["Green", "Yellow", "Red"]
label2id = {label: i for i, label in enumerate(LABEL_NAMES)}
id2label = {i: label for label, i in label2id.items()}

MAX_LENGTH = 160


# =========================
# Load JSONL
# =========================

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


train_rows = load_jsonl(TRAIN_PATH)
val_rows = load_jsonl(VAL_PATH)
test_rows = load_jsonl(TEST_PATH)

print("Train:", len(train_rows))
print("Validation:", len(val_rows))
print("Test:", len(test_rows))


# =========================
# PyTorch Dataset
# =========================

class UrgencyDataset(Dataset):
    def __init__(self, rows, tokenizer, max_length=160):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]

        text = row["text"]
        label = row["labels"]["urgency_id"]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

        return item


# =========================
# Tokenizer and Datasets
# =========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = UrgencyDataset(train_rows, tokenizer, MAX_LENGTH)
val_dataset = UrgencyDataset(val_rows, tokenizer, MAX_LENGTH)
test_dataset = UrgencyDataset(test_rows, tokenizer, MAX_LENGTH)


# =========================
# Model
# =========================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_NAMES),
    id2label=id2label,
    label2id=label2id
)


# =========================
# Metrics
# =========================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, preds)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted
    }


# =========================
# Training Arguments
# =========================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,

    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    report_to="none",
    save_total_limit=2
)


# =========================
# Trainer
# =========================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)


# =========================
# Train
# =========================

trainer.train()


# =========================
# Evaluate on Validation
# =========================

print("\n=== Validation Results ===")
val_results = trainer.evaluate(val_dataset)
print(val_results)


# =========================
# Evaluate on Test
# =========================

print("\n=== Test Results ===")
test_results = trainer.evaluate(test_dataset)
print(test_results)


# =========================
# Detailed Test Report
# =========================

predictions = trainer.predict(test_dataset)

pred_logits = predictions.predictions
pred_labels = np.argmax(pred_logits, axis=1)
true_labels = predictions.label_ids

print("\n=== Classification Report ===")
print(
    classification_report(
        true_labels,
        pred_labels,
        target_names=LABEL_NAMES,
        digits=4,
        zero_division=0
    )
)

print("\n=== Confusion Matrix ===")
print(confusion_matrix(true_labels, pred_labels))


# =========================
# Save Final Model
# =========================

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\nModel saved to: {OUTPUT_DIR}")

Train: 1958
Validation: 420
Test: 420


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



=== Validation Results ===


{'eval_loss': 0.5303138494491577, 'eval_accuracy': 0.7880952380952381, 'eval_precision_macro': 0.7824422959147816, 'eval_recall_macro': 0.7938892523409619, 'eval_f1_macro': 0.7844877844877844, 'eval_precision_weighted': 0.7869836321625425, 'eval_recall_weighted': 0.7880952380952381, 'eval_f1_weighted': 0.7840842412270984, 'eval_runtime': 2.0113, 'eval_samples_per_second': 208.815, 'eval_steps_per_second': 6.961, 'epoch': 3.0}

=== Test Results ===
{'eval_loss': 0.46724364161491394, 'eval_accuracy': 0.8023809523809524, 'eval_precision_macro': 0.7994978417513628, 'eval_recall_macro': 0.8062635806538245, 'eval_f1_macro': 0.8023106395520189, 'eval_precision_weighted': 0.8021344182309977, 'eval_recall_weighted': 0.8023809523809524, 'eval_f1_weighted': 0.801778451433624, 'eval_runtime': 1.875, 'eval_samples_per_second': 224.005, 'eval_steps_per_second': 7.467, 'epoch': 3.0}

=== Classification Report ===
              precision    recall  f1-score   support

       Green     0.8571    0.8415

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to: ./urgency_distilbert_model


In [10]:
import json
import numpy as np
import torch

from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report
)


# =========================
# Configuration
# =========================

MODEL_NAME = "distilbert-base-uncased"

TRAIN_PATH = "train.jsonl"
VAL_PATH = "validation.jsonl"
TEST_PATH = "test.jsonl"

OUTPUT_DIR = "./risk_factor_distilbert_model"

RISK_FACTOR_LABELS = [
    "Chest_Pain_or_Pressure",
    "Respiratory_Distress",
    "Acute_Neurological",
    "Severe_Infection",
    "Anaphylaxis_or_Allergy",
    "Uncontrolled_Bleeding",
    "Severe_Pain",
    "Trauma_or_Injury",
    "Medication_Adverse_Reaction",
    "None"
]

MAX_LENGTH = 160
THRESHOLD = 0.5


# =========================
# Load JSONL
# =========================

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


train_rows = load_jsonl(TRAIN_PATH)
val_rows = load_jsonl(VAL_PATH)
test_rows = load_jsonl(TEST_PATH)

print("Train:", len(train_rows))
print("Validation:", len(val_rows))
print("Test:", len(test_rows))


# =========================
# PyTorch Dataset
# =========================

class RiskFactorDataset(Dataset):
    def __init__(self, rows, tokenizer, max_length=160):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]

        text = row["text"]
        labels = row["labels"]["risk_vector"]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(labels, dtype=torch.float)
        }


# =========================
# Tokenizer and Datasets
# =========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = RiskFactorDataset(train_rows, tokenizer, MAX_LENGTH)
val_dataset = RiskFactorDataset(val_rows, tokenizer, MAX_LENGTH)
test_dataset = RiskFactorDataset(test_rows, tokenizer, MAX_LENGTH)


# =========================
# Model
# =========================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(RISK_FACTOR_LABELS),
    problem_type="multi_label_classification"
)


# =========================
# Metrics
# =========================

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = sigmoid(logits)
    preds = (probs >= THRESHOLD).astype(int)

    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)

    micro_precision = precision_score(labels, preds, average="micro", zero_division=0)
    micro_recall = recall_score(labels, preds, average="micro", zero_division=0)

    macro_precision = precision_score(labels, preds, average="macro", zero_division=0)
    macro_recall = recall_score(labels, preds, average="macro", zero_division=0)

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall
    }


# =========================
# Training Arguments
# =========================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,

    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,

    report_to="none",
    save_total_limit=2
)


# =========================
# Trainer
# =========================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)


# =========================
# Train
# =========================

trainer.train()


# =========================
# Evaluate
# =========================

print("\n=== Validation Results ===")
val_results = trainer.evaluate(val_dataset)
print(val_results)

print("\n=== Test Results ===")
test_results = trainer.evaluate(test_dataset)
print(test_results)


# =========================
# Detailed Test Report
# =========================

predictions = trainer.predict(test_dataset)

logits = predictions.predictions
true_labels = predictions.label_ids

probs = sigmoid(logits)
pred_labels = (probs >= THRESHOLD).astype(int)

print("\n=== Multi-label Classification Report ===")
print(
    classification_report(
        true_labels,
        pred_labels,
        target_names=RISK_FACTOR_LABELS,
        digits=4,
        zero_division=0
    )
)

# Per-label support check
print("\n=== Test Label Supports ===")
supports = true_labels.sum(axis=0)
for label, support in zip(RISK_FACTOR_LABELS, supports):
    print(f"{label}: {int(support)}")


# =========================
# Save Final Model
# =========================

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\nModel saved to: {OUTPUT_DIR}")

Train: 1958
Validation: 420
Test: 420


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Micro Precision,Micro Recall,Macro Precision,Macro Recall
1,0.311244,0.259651,0.372180,0.074717,0.951923,0.231308,0.095192,0.061491
2,0.218514,0.162995,0.719530,0.353867,0.968379,0.572430,0.488022,0.317443
3,0.144247,0.126970,0.832021,0.625279,0.949102,0.740654,0.753571,0.551042
4,0.121086,0.115337,0.850900,0.661301,0.945714,0.773364,0.845327,0.594155


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



=== Validation Results ===


{'eval_loss': 0.1153365969657898, 'eval_micro_f1': 0.8508997429305912, 'eval_macro_f1': 0.6613008892298365, 'eval_micro_precision': 0.9457142857142857, 'eval_micro_recall': 0.7733644859813084, 'eval_macro_precision': 0.8453272244996383, 'eval_macro_recall': 0.5941546207415772, 'eval_runtime': 1.889, 'eval_samples_per_second': 222.342, 'eval_steps_per_second': 7.411, 'epoch': 4.0}

=== Test Results ===
{'eval_loss': 0.12480533123016357, 'eval_micro_f1': 0.8490322580645161, 'eval_macro_f1': 0.7148938974858445, 'eval_micro_precision': 0.94, 'eval_micro_recall': 0.7741176470588236, 'eval_macro_precision': 0.851786068677373, 'eval_macro_recall': 0.6458954252228771, 'eval_runtime': 1.8438, 'eval_samples_per_second': 227.793, 'eval_steps_per_second': 7.593, 'epoch': 4.0}

=== Multi-label Classification Report ===
                             precision    recall  f1-score   support

     Chest_Pain_or_Pressure     1.0000    0.9355    0.9667        31
       Respiratory_Distress     0.9348    0

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to: ./risk_factor_distilbert_model


In [11]:
import json
import numpy as np
import torch

from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)


# =========================
# Configuration
# =========================

MODEL_NAME = "distilbert-base-uncased"

TRAIN_PATH = "train.jsonl"
VAL_PATH = "validation.jsonl"
TEST_PATH = "test.jsonl"

OUTPUT_DIR = "./insufficient_info_distilbert_model"

LABEL_NAMES = ["Sufficient", "Insufficient"]

label2id = {
    "Sufficient": 0,
    "Insufficient": 1
}

id2label = {
    0: "Sufficient",
    1: "Insufficient"
}

MAX_LENGTH = 160


# =========================
# Load JSONL
# =========================

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


train_rows = load_jsonl(TRAIN_PATH)
val_rows = load_jsonl(VAL_PATH)
test_rows = load_jsonl(TEST_PATH)

print("Train:", len(train_rows))
print("Validation:", len(val_rows))
print("Test:", len(test_rows))


# =========================
# PyTorch Dataset
# =========================

class InsufficientInfoDataset(Dataset):
    def __init__(self, rows, tokenizer, max_length=160):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]

        text = row["text"]
        label = row["labels"]["insufficient_id"]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }


# =========================
# Tokenizer and Datasets
# =========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = InsufficientInfoDataset(train_rows, tokenizer, MAX_LENGTH)
val_dataset = InsufficientInfoDataset(val_rows, tokenizer, MAX_LENGTH)
test_dataset = InsufficientInfoDataset(test_rows, tokenizer, MAX_LENGTH)


# =========================
# Model
# =========================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)


# =========================
# Metrics
# =========================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, preds)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0
    )

    precision_insufficient, recall_insufficient, f1_insufficient, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary",
        pos_label=1,
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "precision_insufficient": precision_insufficient,
        "recall_insufficient": recall_insufficient,
        "f1_insufficient": f1_insufficient
    }


# =========================
# Training Arguments
# =========================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,

    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_insufficient",
    greater_is_better=True,

    report_to="none",
    save_total_limit=2
)


# =========================
# Trainer
# =========================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)


# =========================
# Train
# =========================

trainer.train()


# =========================
# Evaluate on Validation
# =========================

print("\n=== Validation Results ===")
val_results = trainer.evaluate(val_dataset)
print(val_results)


# =========================
# Evaluate on Test
# =========================

print("\n=== Test Results ===")
test_results = trainer.evaluate(test_dataset)
print(test_results)


# =========================
# Detailed Test Report
# =========================

predictions = trainer.predict(test_dataset)

pred_logits = predictions.predictions
pred_labels = np.argmax(pred_logits, axis=1)
true_labels = predictions.label_ids

print("\n=== Classification Report ===")
print(
    classification_report(
        true_labels,
        pred_labels,
        target_names=LABEL_NAMES,
        digits=4,
        zero_division=0
    )
)

print("\n=== Confusion Matrix ===")
print(confusion_matrix(true_labels, pred_labels))


# =========================
# Save Final Model
# =========================

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\nModel saved to: {OUTPUT_DIR}")

Train: 1958
Validation: 420
Test: 420


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted,Precision Insufficient,Recall Insufficient,F1 Insufficient
1,0.478028,0.433420,0.804762,0.724565,0.734182,0.729066,0.809217,0.804762,0.806808,0.568627,0.604167,0.585859
2,0.363917,0.385332,0.847619,0.804828,0.728974,0.755271,0.838704,0.847619,0.836881,0.742424,0.510417,0.604938
3,0.239422,0.366552,0.847619,0.785127,0.776620,0.780707,0.845501,0.847619,0.846465,0.673913,0.645833,0.659574


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



=== Validation Results ===


{'eval_loss': 0.3665521442890167, 'eval_accuracy': 0.8476190476190476, 'eval_precision_macro': 0.7851272534464475, 'eval_recall_macro': 0.7766203703703705, 'eval_f1_macro': 0.7807074794413262, 'eval_precision_weighted': 0.8455006817148918, 'eval_recall_weighted': 0.8476190476190476, 'eval_f1_weighted': 0.8464653998918455, 'eval_precision_insufficient': 0.6739130434782609, 'eval_recall_insufficient': 0.6458333333333334, 'eval_f1_insufficient': 0.6595744680851063, 'eval_runtime': 1.8828, 'eval_samples_per_second': 223.067, 'eval_steps_per_second': 7.436, 'epoch': 3.0}

=== Test Results ===
{'eval_loss': 0.28535088896751404, 'eval_accuracy': 0.8690476190476191, 'eval_precision_macro': 0.7886904761904763, 'eval_recall_macro': 0.7967333114825741, 'eval_f1_macro': 0.7925925925925925, 'eval_precision_weighted': 0.8709608843537415, 'eval_recall_weighted': 0.8690476190476191, 'eval_f1_weighted': 0.86994708994709, 'eval_precision_insufficient': 0.6547619047619048, 'eval_recall_insufficient': 0.6

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to: ./insufficient_info_distilbert_model


In [12]:
from google.colab import drive
import shutil
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define the backup path on Drive
backup_path = '/content/drive/MyDrive/Colab_Models_Backup'
if not os.path.exists(backup_path):
    os.makedirs(backup_path)
    print(f"Created backup folder: {backup_path}")

# 3. List of items to back up (Models and Datasets)
items_to_backup = [
    'urgency_distilbert_model',
    'risk_factor_distilbert_model',
    'insufficient_info_distilbert_model',
    'train.jsonl',
    'validation.jsonl',
    'test.jsonl',
    'final_synthetic_dataset_local_llm.jsonl'
]

for item in items_to_backup:
    source = f'./{item}'
    destination = os.path.join(backup_path, item)

    if os.path.exists(source):
        if os.path.isdir(source):
            # Copy directory
            if os.path.exists(destination):
                shutil.rmtree(destination) # Overwrite if exists
            shutil.copytree(source, destination)
            print(f"Successfully backed up folder: {item}")
        else:
            # Copy file
            shutil.copy2(source, destination)
            print(f"Successfully backed up file: {item}")
    else:
        print(f"Skipping {item}: Not found in local directory (did you run the training cells yet?)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Created backup folder: /content/drive/MyDrive/Colab_Models_Backup
Successfully backed up folder: urgency_distilbert_model
Successfully backed up folder: risk_factor_distilbert_model
Successfully backed up folder: insufficient_info_distilbert_model
Successfully backed up file: train.jsonl
Successfully backed up file: validation.jsonl
Successfully backed up file: test.jsonl
Successfully backed up file: final_synthetic_dataset_local_llm.jsonl


In [14]:
import json
import os
from pathlib import Path

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification


# =========================
# Model paths (Fixed for Colab)
# =========================

URGENCY_MODEL_DIR = "./urgency_distilbert_model"
RISK_MODEL_DIR = "./risk_factor_distilbert_model"
INSUFFICIENT_MODEL_DIR = "./insufficient_info_distilbert_model"

MAX_LENGTH = 160
RISK_FACTOR_THRESHOLD = 0.30

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


URGENCY_LABELS = ["Green", "Yellow", "Red"]

RISK_FACTOR_LABELS = [
    "Chest_Pain_or_Pressure",
    "Respiratory_Distress",
    "Acute_Neurological",
    "Severe_Infection",
    "Anaphylaxis_or_Allergy",
    "Uncontrolled_Bleeding",
    "Severe_Pain",
    "Trauma_or_Injury",
    "Medication_Adverse_Reaction",
    "None"
]

INSUFFICIENT_LABELS = ["Sufficient", "Insufficient"]


# =========================
# Path helpers
# =========================

def find_model_folder(path_str):
    path = Path(path_str)

    if path.suffix.lower() == ".zip":
        raise ValueError(
            f"\nPath is a ZIP file, not a directory:\n{path}\n"
            "Please unzip the file first."
        )

    if not path.exists():
        raise FileNotFoundError(f"\nPath does not exist:\n{path}")

    if not path.is_dir():
        raise NotADirectoryError(f"\nPath exists but is not a directory:\n{path}")

    config_file = path / "config.json"
    model_file_1 = path / "model.safetensors"
    model_file_2 = path / "pytorch_model.bin"

    if config_file.exists() and (model_file_1.exists() or model_file_2.exists()):
        return str(path)

    for root, dirs, files in os.walk(path):
        files_set = set(files)
        if "config.json" in files_set and (
            "model.safetensors" in files_set or "pytorch_model.bin" in files_set
        ):
            return root

    raise FileNotFoundError(
        f"\nValid model folder not found inside:\n{path}"
    )


def check_model_folder(name, path_str):
    print(f"\nChecking {name} model path...")
    model_path = find_model_folder(path_str)
    print(f"Model folder found: {model_path}")
    return model_path


# =========================
# Resolve folders
# =========================

URGENCY_MODEL_DIR = check_model_folder("urgency", URGENCY_MODEL_DIR)
RISK_MODEL_DIR = check_model_folder("risk factor", RISK_MODEL_DIR)
INSUFFICIENT_MODEL_DIR = check_model_folder("insufficient info", INSUFFICIENT_MODEL_DIR)


# =========================
# Load models
# =========================

def load_classifier_model(model_dir, name):
    print(f"\nLoading {name} model from: {model_dir}")
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.to(DEVICE)
    model.eval()
    print(f"{name} model loaded successfully.")
    return tokenizer, model


urgency_tokenizer, urgency_model = load_classifier_model(URGENCY_MODEL_DIR, "urgency")
risk_tokenizer, risk_model = load_classifier_model(RISK_MODEL_DIR, "risk factor")
insufficient_tokenizer, insufficient_model = load_classifier_model(INSUFFICIENT_MODEL_DIR, "insufficient info")

print("\nAll models loaded successfully.")
print("Device:", DEVICE)


# =========================
# Math helpers
# =========================

def softmax(logits):
    logits = logits - np.max(logits)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum()


def sigmoid(logits):
    return 1 / (1 + np.exp(-logits))


# =========================
# Prediction functions
# =========================

def predict_single_class(text, tokenizer, model, labels):
    encoding = tokenizer(text, truncation=True, padding="max_length", max_length=MAX_LENGTH, return_tensors="pt")
    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    logits = outputs.logits[0].detach().cpu().numpy()
    probs = softmax(logits)

    pred_id = int(np.argmax(probs))
    return labels[pred_id], float(probs[pred_id]), {labels[i]: float(probs[i]) for i in range(len(labels))}


def predict_risk_factors(text):
    encoding = risk_tokenizer(text, truncation=True, padding="max_length", max_length=MAX_LENGTH, return_tensors="pt")
    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    with torch.no_grad():
        outputs = risk_model(input_ids=input_ids, attention_mask=attention_mask)

    logits = outputs.logits[0].detach().cpu().numpy()
    probs = sigmoid(logits)

    detected = [RISK_FACTOR_LABELS[i] for i, p in enumerate(probs) if p >= RISK_FACTOR_THRESHOLD]
    if not detected or (len(detected) == 1 and detected[0] == "None"):
        best_idx = int(np.argmax(probs))
        detected = [RISK_FACTOR_LABELS[best_idx]]

    return detected, {RISK_FACTOR_LABELS[i]: float(p) for i, p in enumerate(probs)}


def determine_routing(urgency_label, risk_factors, insufficient_label):
    insufficient_flag = insufficient_label == "Insufficient"
    if urgency_label == "Red": return {"needs_human_review": True, "target_action": "Immediate human review"}
    if urgency_label == "Yellow": return {"needs_human_review": True, "target_action": "Same-day human review"}
    if insufficient_flag or any(r != "None" for r in risk_factors):
        return {"needs_human_review": True, "target_action": "Human review required"}
    return {"needs_human_review": False, "target_action": "Routine queue"}


# =========================
# Unified inference
# =========================

def analyze_patient_message(message):
    u_label, u_conf, u_probs = predict_single_class(message, urgency_tokenizer, urgency_model, URGENCY_LABELS)
    r_factors, r_probs = predict_risk_factors(message)
    i_label, i_conf, i_probs = predict_single_class(message, insufficient_tokenizer, insufficient_model, INSUFFICIENT_LABELS)

    routing = determine_routing(urgency_label=u_label, risk_factors=r_factors, insufficient_label=i_label)
    return {
        "input_message": message,
        "analysis": {
            "urgency": u_label,
            "urgency_confidence": round(u_conf, 4),
            "risk_factors": r_factors,
            "insufficient": i_label
        },
        "routing": routing
    }


# =========================
# Example messages
# =========================

example_messages = [
    "I have had a sore throat for two days and wanted to ask if I should schedule an appointment.",
    "My dad says he has pressure in his chest and feels short of breath.",
    "I feel weird and bad since yesterday but I do not know how to explain it.",
    "I need a refill for my allergy medication.",
    "My child fell yesterday and now has a swollen ankle and a lot of pain."
]

for msg in example_messages:
    output = analyze_patient_message(msg)
    print("\n" + "=" * 60)
    print(f"MESSAGE: {msg}")
    print(f"OUTPUT: {json.dumps(output, indent=2)}")


Checking urgency model path...
Model folder found: urgency_distilbert_model

Checking risk factor model path...
Model folder found: risk_factor_distilbert_model

Checking insufficient info model path...
Model folder found: insufficient_info_distilbert_model

Loading urgency model from: urgency_distilbert_model


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

urgency model loaded successfully.

Loading risk factor model from: risk_factor_distilbert_model


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

risk factor model loaded successfully.

Loading insufficient info model from: insufficient_info_distilbert_model


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

insufficient info model loaded successfully.

All models loaded successfully.
Device: cuda

MESSAGE: I have had a sore throat for two days and wanted to ask if I should schedule an appointment.
OUTPUT: {
  "input_message": "I have had a sore throat for two days and wanted to ask if I should schedule an appointment.",
  "analysis": {
    "urgency": "Green",
    "urgency_confidence": 0.9124,
    "risk_factors": [
      "None"
    ],
    "insufficient": "Sufficient"
  },
  "routing": {
    "needs_human_review": false,
    "target_action": "Routine queue"
  }
}

MESSAGE: My dad says he has pressure in his chest and feels short of breath.
OUTPUT: {
  "input_message": "My dad says he has pressure in his chest and feels short of breath.",
  "analysis": {
    "urgency": "Red",
    "urgency_confidence": 0.7244,
    "risk_factors": [
      "Chest_Pain_or_Pressure",
      "Respiratory_Distress"
    ],
    "insufficient": "Insufficient"
  },
  "routing": {
    "needs_human_review": true,
    "targ

In [15]:
import json
import csv
import os
from pathlib import Path

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification


# =========================
# Configuration
# =========================

URGENCY_MODEL_DIR = r"urgency_distilbert_model"
RISK_MODEL_DIR = r"risk_factor_distilbert_model"
INSUFFICIENT_MODEL_DIR = r"insufficient_info_distilbert_model"

TEST_PATH = "test.jsonl"

OUTPUT_JSONL = "pipeline_test_outputs.jsonl"
OUTPUT_CSV = "pipeline_test_outputs.csv"

# כמה דוגמאות להריץ מתוך test.
# אפשר לשנות ל-None כדי להריץ על כל ה-test set.
NUM_EXAMPLES = 50

MAX_LENGTH = 160
RISK_FACTOR_THRESHOLD = 0.30

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

URGENCY_LABELS = ["Green", "Yellow", "Red"]

RISK_FACTOR_LABELS = [
    "Chest_Pain_or_Pressure",
    "Respiratory_Distress",
    "Acute_Neurological",
    "Severe_Infection",
    "Anaphylaxis_or_Allergy",
    "Uncontrolled_Bleeding",
    "Severe_Pain",
    "Trauma_or_Injury",
    "Medication_Adverse_Reaction",
    "None"
]

INSUFFICIENT_LABELS = ["Sufficient", "Insufficient"]


# =========================
# Load JSONL
# =========================

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


test_rows = load_jsonl(TEST_PATH)

if NUM_EXAMPLES is not None:
    test_rows = test_rows[:NUM_EXAMPLES]

print("Test examples to analyze:", len(test_rows))


# =========================
# Path helper
# =========================

def find_model_folder(path_str):
    path = Path(path_str)

    if not path.exists():
        raise FileNotFoundError(f"Path does not exist: {path}")

    if not path.is_dir():
        raise NotADirectoryError(f"Path is not a directory: {path}")

    if (path / "config.json").exists() and (
        (path / "model.safetensors").exists() or (path / "pytorch_model.bin").exists()
    ):
        return str(path)

    for root, dirs, files in os.walk(path):
        files_set = set(files)
        if "config.json" in files_set and (
            "model.safetensors" in files_set or "pytorch_model.bin" in files_set
        ):
            return root

    raise FileNotFoundError(f"No valid model folder found inside: {path}")


URGENCY_MODEL_DIR = find_model_folder(URGENCY_MODEL_DIR)
RISK_MODEL_DIR = find_model_folder(RISK_MODEL_DIR)
INSUFFICIENT_MODEL_DIR = find_model_folder(INSUFFICIENT_MODEL_DIR)


# =========================
# Load models
# =========================

def load_classifier_model(model_dir, name):
    print(f"Loading {name} model from: {model_dir}")

    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)

    model.to(DEVICE)
    model.eval()

    print(f"{name} model loaded.")
    return tokenizer, model


urgency_tokenizer, urgency_model = load_classifier_model(
    URGENCY_MODEL_DIR,
    "urgency"
)

risk_tokenizer, risk_model = load_classifier_model(
    RISK_MODEL_DIR,
    "risk factor"
)

insufficient_tokenizer, insufficient_model = load_classifier_model(
    INSUFFICIENT_MODEL_DIR,
    "insufficient information"
)

print("All models loaded.")
print("Device:", DEVICE)


# =========================
# Math helpers
# =========================

def softmax(logits):
    logits = logits - np.max(logits)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum()


def sigmoid(logits):
    return 1 / (1 + np.exp(-logits))


# =========================
# Prediction functions
# =========================

def predict_single_class(text, tokenizer, model, labels):
    encoding = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    logits = outputs.logits[0].detach().cpu().numpy()
    probs = softmax(logits)

    pred_id = int(np.argmax(probs))
    pred_label = labels[pred_id]
    confidence = float(probs[pred_id])

    all_probs = {
        labels[i]: float(probs[i])
        for i in range(len(labels))
    }

    return pred_label, confidence, all_probs


def predict_risk_factors(text):
    encoding = risk_tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    with torch.no_grad():
        outputs = risk_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    logits = outputs.logits[0].detach().cpu().numpy()
    probs = sigmoid(logits)

    detected = []
    probabilities = {}

    for label, prob in zip(RISK_FACTOR_LABELS, probs):
        prob_float = float(prob)
        probabilities[label] = prob_float

        if prob_float >= RISK_FACTOR_THRESHOLD:
            detected.append(label)

    non_none_detected = [
        label for label in detected
        if label != "None"
    ]

    if len(non_none_detected) > 0:
        detected = non_none_detected
    elif "None" in detected:
        detected = ["None"]
    else:
        best_idx = int(np.argmax(probs))
        detected = [RISK_FACTOR_LABELS[best_idx]]

    return detected, probabilities


def determine_routing(urgency_label, risk_factors, insufficient_label):
    """
    Administrative routing only.
    Not diagnosis, treatment, or medical advice.
    """

    insufficient_flag = insufficient_label == "Insufficient"
    risk_without_none = [r for r in risk_factors if r != "None"]

    if urgency_label == "Red":
        return True, "Immediate human review"

    if urgency_label == "Yellow":
        return True, "Same-day human review"

    if insufficient_flag:
        return True, "Human review required due to insufficient information"

    if len(risk_without_none) > 0:
        return True, "Human review required due to detected risk signal"

    return False, "Routine queue"


def analyze_patient_message(message):
    urgency_label, urgency_confidence, urgency_probs = predict_single_class(
        message,
        urgency_tokenizer,
        urgency_model,
        URGENCY_LABELS
    )

    risk_factors, risk_probs = predict_risk_factors(message)

    insufficient_label, insufficient_confidence, insufficient_probs = predict_single_class(
        message,
        insufficient_tokenizer,
        insufficient_model,
        INSUFFICIENT_LABELS
    )

    needs_human_review, target_action = determine_routing(
        urgency_label,
        risk_factors,
        insufficient_label
    )

    return {
        "input_message": message,
        "analysis": {
            "urgency": urgency_label,
            "urgency_confidence": round(urgency_confidence, 4),
            "risk_factors": risk_factors,
            "insufficient": insufficient_label,
            "insufficient_confidence": round(insufficient_confidence, 4)
        },
        "routing": {
            "needs_human_review": needs_human_review,
            "target_action": target_action
        },
        "debug_probabilities": {
            "urgency_probabilities": {
                k: round(v, 4) for k, v in urgency_probs.items()
            },
            "risk_factor_probabilities": {
                k: round(v, 4) for k, v in risk_probs.items()
            },
            "insufficient_probabilities": {
                k: round(v, 4) for k, v in insufficient_probs.items()
            }
        }
    }


# =========================
# Run pipeline on test examples
# =========================

outputs = []

for i, row in enumerate(test_rows):
    message = row["text"]
    true_labels = row["labels"]

    prediction = analyze_patient_message(message)

    record = {
        "example_id": i,
        "input_message": message,

        "true_urgency": true_labels["urgency_level"],
        "pred_urgency": prediction["analysis"]["urgency"],
        "urgency_confidence": prediction["analysis"]["urgency_confidence"],

        "true_risk_factors": [
            label for label, value in zip(RISK_FACTOR_LABELS, true_labels["risk_vector"])
            if value == 1
        ],
        "pred_risk_factors": prediction["analysis"]["risk_factors"],

        "true_insufficient": bool(true_labels["insufficient_information"]),
        "pred_insufficient": prediction["analysis"]["insufficient"] == "Insufficient",
        "insufficient_confidence": prediction["analysis"]["insufficient_confidence"],

        "needs_human_review": prediction["routing"]["needs_human_review"],
        "target_action": prediction["routing"]["target_action"],

        "full_output": prediction
    }

    outputs.append(record)

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(test_rows)}")


# =========================
# Save JSONL
# =========================

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for record in outputs:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("\nSaved JSONL to:", OUTPUT_JSONL)


# =========================
# Save CSV
# =========================

csv_fields = [
    "example_id",
    "input_message",
    "true_urgency",
    "pred_urgency",
    "urgency_confidence",
    "true_risk_factors",
    "pred_risk_factors",
    "true_insufficient",
    "pred_insufficient",
    "insufficient_confidence",
    "needs_human_review",
    "target_action"
]

with open(OUTPUT_CSV, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=csv_fields)
    writer.writeheader()

    for record in outputs:
        row = {
            "example_id": record["example_id"],
            "input_message": record["input_message"],
            "true_urgency": record["true_urgency"],
            "pred_urgency": record["pred_urgency"],
            "urgency_confidence": record["urgency_confidence"],
            "true_risk_factors": "; ".join(record["true_risk_factors"]),
            "pred_risk_factors": "; ".join(record["pred_risk_factors"]),
            "true_insufficient": record["true_insufficient"],
            "pred_insufficient": record["pred_insufficient"],
            "insufficient_confidence": record["insufficient_confidence"],
            "needs_human_review": record["needs_human_review"],
            "target_action": record["target_action"]
        }

        writer.writerow(row)

print("Saved CSV to:", OUTPUT_CSV)


# =========================
# Print examples for report / slides
# =========================

print("\n\n=== Example outputs for report/slides ===")

for record in outputs[:8]:
    print("\n" + "=" * 80)
    print("Example ID:", record["example_id"])
    print("Message:")
    print(record["input_message"])

    print("\nTrue labels:")
    print("Urgency:", record["true_urgency"])
    print("Risk factors:", record["true_risk_factors"])
    print("Insufficient:", record["true_insufficient"])

    print("\nPredicted output:")
    print("Urgency:", record["pred_urgency"], "| confidence:", record["urgency_confidence"])
    print("Risk factors:", record["pred_risk_factors"])
    print("Insufficient:", record["pred_insufficient"], "| confidence:", record["insufficient_confidence"])
    print("Routing:", record["target_action"])

Test examples to analyze: 50
Loading urgency model from: urgency_distilbert_model


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

urgency model loaded.
Loading risk factor model from: risk_factor_distilbert_model


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

risk factor model loaded.
Loading insufficient information model from: insufficient_info_distilbert_model


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

insufficient information model loaded.
All models loaded.
Device: cuda
Processed 10/50
Processed 20/50
Processed 30/50
Processed 40/50
Processed 50/50

Saved JSONL to: pipeline_test_outputs.jsonl
Saved CSV to: pipeline_test_outputs.csv


=== Example outputs for report/slides ===

Example ID: 0
Message:
OMG my tongue just started swelling up like crazy! And my eyes are watering so much, I can't even see straight! Can someone call me? I feel like I'm going to pass out from this allergy attack!

True labels:
Urgency: Red
Risk factors: ['Anaphylaxis_or_Allergy']
Insufficient: False

Predicted output:
Urgency: Red | confidence: 0.8081
Risk factors: ['Anaphylaxis_or_Allergy']
Insufficient: False | confidence: 0.9767
Routing: Immediate human review

Example ID: 1
Message:
I'm super worried about this rash on my arm. It's been really itchy and it keeps getting bigger. I feel like something bad might be happening. Should I just ignore it or should I go see a doctor?

True labels:
Urgency: Yello

In [16]:
import json
from collections import Counter
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# =========================
# Configuration
# =========================

INPUT_JSONL = "pipeline_test_outputs.jsonl"

URGENCY_LABELS = ["Green", "Yellow", "Red"]


# =========================
# Load pipeline outputs
# =========================

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


rows = load_jsonl(INPUT_JSONL)

print("Loaded examples:", len(rows))


# =========================
# Collect labels
# =========================

true_urgency = []
pred_urgency = []

true_insufficient = []
pred_insufficient = []

risk_exact_matches = []
human_review_flags = []
target_actions = []

for row in rows:
    true_urgency.append(row["true_urgency"])
    pred_urgency.append(row["pred_urgency"])

    true_insufficient.append(row["true_insufficient"])
    pred_insufficient.append(row["pred_insufficient"])

    true_risks = set(row["true_risk_factors"])
    pred_risks = set(row["pred_risk_factors"])

    risk_exact_matches.append(true_risks == pred_risks)

    human_review_flags.append(row["needs_human_review"])
    target_actions.append(row["target_action"])


# =========================
# Urgency metrics
# =========================

print("\n=== Urgency Classification ===")

urgency_accuracy = accuracy_score(true_urgency, pred_urgency)

print("Urgency Accuracy:", round(urgency_accuracy, 4))

print("\nUrgency Classification Report:")
print(
    classification_report(
        true_urgency,
        pred_urgency,
        labels=URGENCY_LABELS,
        digits=4,
        zero_division=0
    )
)

print("\nUrgency Confusion Matrix:")
print(URGENCY_LABELS)
print(confusion_matrix(true_urgency, pred_urgency, labels=URGENCY_LABELS))


# =========================
# Risk factor exact match
# =========================

print("\n=== Risk Factor Prediction ===")

risk_exact_accuracy = sum(risk_exact_matches) / len(risk_exact_matches)

print("Risk factor exact-match accuracy:", round(risk_exact_accuracy, 4))

print("\nMost common true risk sets:")
true_risk_counter = Counter(
    tuple(sorted(row["true_risk_factors"]))
    for row in rows
)

for risks, count in true_risk_counter.most_common(10):
    print(risks, ":", count)

print("\nMost common predicted risk sets:")
pred_risk_counter = Counter(
    tuple(sorted(row["pred_risk_factors"]))
    for row in rows
)

for risks, count in pred_risk_counter.most_common(10):
    print(risks, ":", count)


# =========================
# Insufficient information metrics
# =========================

print("\n=== Insufficient Information ===")

insufficient_accuracy = accuracy_score(true_insufficient, pred_insufficient)

print("Insufficient accuracy:", round(insufficient_accuracy, 4))

print("\nInsufficient Classification Report:")
print(
    classification_report(
        true_insufficient,
        pred_insufficient,
        target_names=["Sufficient", "Insufficient"],
        digits=4,
        zero_division=0
    )
)

print("\nInsufficient Confusion Matrix:")
print(confusion_matrix(true_insufficient, pred_insufficient))


# =========================
# Routing summary
# =========================

print("\n=== Routing Summary ===")

review_count = sum(human_review_flags)
routine_count = len(human_review_flags) - review_count

print("Needs human review:", review_count)
print("Routine queue:", routine_count)
print("Human review rate:", round(review_count / len(rows), 4))

print("\nTarget action distribution:")
for action, count in Counter(target_actions).most_common():
    print(action, ":", count)


# =========================
# Error examples
# =========================

print("\n=== Example Errors ===")

error_count = 0

for row in rows:
    urgency_error = row["true_urgency"] != row["pred_urgency"]
    risk_error = set(row["true_risk_factors"]) != set(row["pred_risk_factors"])
    insufficient_error = row["true_insufficient"] != row["pred_insufficient"]

    if urgency_error or risk_error or insufficient_error:
        error_count += 1

        print("\n" + "=" * 80)
        print("Example ID:", row["example_id"])
        print("Message:")
        print(row["input_message"])

        print("\nTrue:")
        print("Urgency:", row["true_urgency"])
        print("Risk:", row["true_risk_factors"])
        print("Insufficient:", row["true_insufficient"])

        print("\nPredicted:")
        print("Urgency:", row["pred_urgency"])
        print("Risk:", row["pred_risk_factors"])
        print("Insufficient:", row["pred_insufficient"])
        print("Routing:", row["target_action"])

        print("\nError types:")
        if urgency_error:
            print("- Urgency error")
        if risk_error:
            print("- Risk factor error")
        if insufficient_error:
            print("- Insufficient information error")

        if error_count >= 8:
            break

print("\nTotal examples with at least one error:", error_count)

Loaded examples: 50

=== Urgency Classification ===
Urgency Accuracy: 0.76

Urgency Classification Report:
              precision    recall  f1-score   support

       Green     0.8333    0.8000    0.8163        25
      Yellow     0.6250    0.6667    0.6452        15
         Red     0.8000    0.8000    0.8000        10

    accuracy                         0.7600        50
   macro avg     0.7528    0.7556    0.7538        50
weighted avg     0.7642    0.7600    0.7617        50


Urgency Confusion Matrix:
['Green', 'Yellow', 'Red']
[[20  5  0]
 [ 3 10  2]
 [ 1  1  8]]

=== Risk Factor Prediction ===
Risk factor exact-match accuracy: 0.92

Most common true risk sets:
('None',) : 21
('Acute_Neurological',) : 6
('Medication_Adverse_Reaction',) : 6
('Severe_Pain',) : 5
('Respiratory_Distress',) : 4
('Severe_Infection',) : 3
('Anaphylaxis_or_Allergy',) : 2
('Trauma_or_Injury',) : 2
('Chest_Pain_or_Pressure',) : 1

Most common predicted risk sets:
('None',) : 21
('Severe_Pain',) : 7
('Me